In [1]:
import numpy as np
from pathlib import Path
import timeit
import pandas as pd
import util
import scipy as sp

In [2]:
def update_equation(D,A=None,pc=None,pr=None,order='ST'):
    """ 
    update_equation updates the probability(ies) to be degraded, D, by one time step, 
    given the interaction matrix A, collapse rate pc and recovery rate pr

    Parameters
    ---
    D : numpy.array
        The initial array of degraded probabilities
    A : numpy.array
        The interaction matrix (i.e. adjacency matrix), must be of shape (len(D), len(D)). 
        No self loops, i.e. all diagonal elements should be 0.
        Default is np.zeros([len(D)]*2) i.e. the empty graph, such that all Markov chains are disconnected.
    pc : numpy.array or something broadcastable
        The array/list/collection of collapse rates. Must be in [0,1]. 
        If none supplied, default is 1/2 for all Markov chains
    pr : numpy.array or something broadcastable
        The array/list/collection of collapse rates. Must be in [0,1]
        If none supplied default is tr = 10tc where tr is the recovery time and 
        tc the collapse time with pc=1/(tc+1)
    order : str
        The order of the interaction matrix, 
        if 'ST' then the interaction matrix is indexed as A[source, target]
        if 'TS' then the interaction matrix is indexed as A[target, source]
        Default is 'ST'
        
    Returns
    ---
    D_out : numpy.array

    Examples
    ---
    For an initial vector of zeros, regardless of the interaction matrix, D_out will always be pc
    >>> update_equation(np.zeros(2), A = np.array( [[0, -2], [4, 0]] ), pc = np.array([0.2, 0.9]), order='ST')
    array([0.2, 0.9]) 
    
    For an initial vector of ones, regardless of the interaction matrix, D_out will always be pr
    >>> update_equation(np.ones(2), A = np.array( [[0, -2], [4, 0]] ), pc = np.array([0.2, 0.9]), pr = np.array([0.5, 0.5]), order='ST')
    array([0.5, 0.5])

    For non-trivial initial D:
    >>> update_equation(np.array([0, 1]), A = np.array( [[0, -2], [4, 0]] ), pc = np.array([0.2, 0.9]), 
        >>> pr = np.array([0.5, 0.5]), order='ST')
    array([1, 0.5])
    >>> update_equation_quick(np.array([0.5, 0.5]), np.array( [[0, 1], [1, 0]] ), np.array([0.2, 0.45]), np.array([0.6, 0.3]))
    array([0.35  , 0.6875])
    """
    if A is None:
        A = np.zeros([len(D)]*2)
    elif order == 'ST':
        A = A.transpose()
    if pc is None:
        pc = np.ones(len(D))/2
    if pr is None:
        pr = 1/(10/pc - 9)

    ### Decomposing interaction matrix as A = A_pos + A_neg, 
        ### where A_pos contains all positive elements of A, while A_neg contains all negatives
    A_pos = np.maximum(A,0)
    A_neg = np.maximum(-A,0)
    d_pos = np.dot(A_pos,D)
    d_neg = np.dot(A_neg,D)

    return (1+d_pos)*pc*(1-D)/(1+d_neg) + (1-pr)*D

def update_equation_quick(D,A,pc,pr):
    """ 
    update_equation updates the probability(ies) to be degraded, D, by one time step, 
    given the interaction matrix A, collapse rate pc and recovery rate pr. 
    Quicker version to go into wrappers and other fns

    Parameters
    ---
    D : numpy.array
        The initial array of degraded probabilities
    A : numpy.array
        The interaction matrix (i.e. adjacency matrix), must be of shape (len(D), len(D)). 
        No self loops, i.e. all diagonal elements should be 0. Must be indexed as A[target,source]
    pc : numpy.array or something broadcastable
        The array/list/collection of collapse rates. Must be in [0,1]. 
    pr : numpy.array or something broadcastable
        The array/list/collection of collapse rates. Must be in [0,1]
        
    Returns
    ---
    D_out : numpy.array

    Examples
    ---
    For an initial vector of zeros, regardless of the interaction matrix, D_out will always be pc
    >>> update_equation(np.zeros(2), np.array( [[0, 4], [-2, 0]] ),np.array([0.2, 0.9]))
    np.array([0.2, 0.9]) 

    For numnerical examples see update_equation
    """
    return (1+np.dot(np.maximum(A,0),D))*pc*(1-D)/(1+np.dot(np.maximum(-A,0),D)) + (1-pr)*D

def temp_response(T,T_lim,beta):
    """ 
    Returns the response in collapse rate to a temperature T, 
    for a tipping element with threshold T_lim and temperature sensitivity (i.e. logistic parameter) beta
    
    Parameters
    ---
    T : numpy.array or float
        Temperature(s)
    T_lim : numpy.array or float
        Tipping/critical threshold(s)
    beta : numpy.array or something broadcastable
        Temperature sensitivity(ies) (i.e. logistic parameter)

    Returns
    ---
    out : numpy.array
    """
    return 1/(1+np.exp(beta*(T_lim-T)))

def evolve(D_init, A, no_time_steps, 
           order_ST=True, 
           pc = None, tau_c = None, 
           T = None, beta = None, T_lim = None,
           pr = None, tau_r = None,
           ):
    """ 
    Wrapper to evolve system up to and including t = no_time_steps, 
    with an initial value of D_init at time t=0. 

    Parameters (required)
    ---
    D_init : numpy.array
        The initial array of degraded probabilities at time step 0
    A : numpy.array
        The interaction matrix (i.e. adjacency matrix), must be of shape (len(D_init), len(D_init)). 
        No self loops, i.e. all diagonal elements should be 0. 
    no_time_steps : int
        The number of time steps to simulate over, 
        i.e. the number of times the update equation is successively applied. 
        Must be >= 1
    
    Parameters (optional):
    order_ST : Bool
        The indexing order of the adjacent matrix. 
        If True, then the adjacency matrix will be read with an index-order that gives A[source, target].
        Otherwise the order will be assumed as A[target, source]
        Default True.
    pc : numpy.array or something broadcastable
        The array/list/collection of collapse rates (without interactions). 
        Will supercede any calls to tau_c or T
        Must be in [0,1]. 
    tau_c :  numpy.array or something broadcastable
        The array/list/collection of collapse time scales. 
    T : array-like or float
        The global temperature(s) at which to evaluate. 
        Must be of length >= no_time_steps. Must have beta and T_lim supplied as well if not None
    beta : array-like or float
        Temperature sensitivities (i.e. logistic parameters) for individual tipping points.
        If array-like must be of length len(D_init). 
    T_lim : array-like or float
        Critical temperature thresholds for individual tipping points.
        If array-like must be of length len(D_init)
    pr : numpy.array or something broadcastable
        The array/list/collection of collapse rates. Will supercede tau_r
    tau_r : numpy.array or something broadcastable
        The array/list/collection of collapse time scales
    
        
    Returns
    ---
    D_out : numpy.array
        The output will be a (len(D_init), no_time_steps+1)-shaped array where the D_out[:,0] = D_init
    Examples
    ---
    """
    ## Initialising D_out
    D_out = np.zeros((len(D_init), no_time_steps))
    D_out[:,0] = D_init
    
    ## Changing pc if needed
    if pc is None:
        if tau_c is None:
            raise ValueError("No collapse time scales given")
            pc = np.ones(len(D_init))/2 ## if nothing is supplied pc will be set to 1/2
        elif (T is None) or (beta is None) or (T_lim is None):
            # if any of T,T_lim,beta are None, then pc = 1/(tau_c+1)
            pc = np.ones((len(D_init), no_time_steps))/(tau_c.reshape(len(D_init),1)+1)
            # print('Non-critical error: at least one of T, T_lim or beta has not been provided.')
        else:
            pc = np.ones((len(D_init), no_time_steps))
            pc /= (tau_c.reshape(len(D_init),1)+1)
            for t in range(no_time_steps):
                pc[:,t] *= temp_response(T[t], T_lim, beta)
    if np.any(pc>1) or np.any(pc<0):
        raise ValueError("Collapse probability pc not in [0,1]")
    ## Changing pr if needed
    if pr is None:
        if tau_r is None:  
            raise ValueError("No recovery time scales given")
        else:
            pr = 1/(tau_r+1)
    if np.any(pr>1) or np.any(pr<0):
        raise ValueError("Recovery probability pr not in [0,1]")
    # print(pr.shape)
    if order_ST:
        A = A.transpose()

    for t in range(1,no_time_steps):
        D_out[:,t] = update_equation_quick(D_out[:,t-1],A,pc[:,t-1],pr)

    return D_out

In [10]:
def stat_dist_theory_res(D,*data):
    A,q_c,q_r = data
    x_pos = np.dot(np.transpose(np.where(A>=0,A,0)),D)
    x_neg = np.dot(np.transpose(np.where(A<0,-A,0)),D)
    # out = np.zeros(len(D))
    # for i in range(len(D)):
    #     out[i] = D[i] - (1+x_pos[i])*q_c[i]/((1+x_pos)*q_c[i] + (1+x_neg)*q_r[i])
    out = D - (1+x_pos)*q_c/((1+x_pos)*q_c + (1+x_neg)*q_r)
    return out

def stat_dist_theory(A,q_c,q_r,T,T_lim,beta):
    return sp.optimize.fsolve(stat_dist_theory_res, np.ones(9), args=(A,temp_response(T, T_lim,beta)*q_c,q_r))

# Testing

## Reading individual TE data

In [4]:
folder = '../Data/'
# df_AM22 = pd.read_excel(folder+"Armstrong McKay 2022/science.abn7950_data_s1.xlsx",
#                         skiprows=[0,1,2,3,4],sheet_name='4) Estimates',
#                         skipfooter=40,index_col=1,header=[0,1])
df_TEs = pd.read_excel(folder+'TEsummary.xlsx',header=[0],index_col=0)

df_TEs.head()

,Tipping element,Collapse (years),Recovery (years),Threshold (°C) – Min.,Threshold (°C) – Est.,Threshold (°C) – Max.,Beta (1/°C),Max Impact (°C) – Global,Comments
TE,,,,,,,,,
AMAZ,Amazon rainforest dieback,100,1000,2.0,3.5,6.0,4.604503,0.20,"Max impact for a total die back is 0.2C, while..."
AMOC,Atlantic meridional ocean circulation,50,500,1.4,4.0,8.0,2.656444,-0.50,NaN
ASI,Arctic winter sea ice,20,200,4.5,6.3,8.7,3.837086,0.60,NaN
ENSO,El Nino Southern Oscillation,100,1000,3.0,4.5,6.0,4.604503,0.00,From Armstrong McKay et al no estimate for max...
GIS,Greenland ice sheet,10000,100000,0.8,1.5,3.0,9.866793,0.13,NaN


## Reading generated interaction matrices

In [6]:
folder = "../Data/Interaction matrices/"
no_samples = 500
A = [np.loadtxt(folder+'A_'+str(r)+'.csv') for r in range(no_samples)]

In [ ]:
D = np.zeros(9)
T = +1.5
D_out = update_equation_quick(D,A[0],
                              temp_response(T,df_TEs['Threshold (°C) – Est.'].to_numpy(), 
                                            df_TEs['Beta (1/°C)']).to_numpy()/ (df_TEs['Collapse (years)'].to_numpy() + 1),
                              1/(df_TEs['Recovery (years)'].to_numpy()+1))

## Reading extended ssps

In [ ]:
df = pd.read_excel('../Data/extendedSSPs.xlsx')
scenarios = df['scenario'].unique()

dfs = {s: df[df['scenario']==s].set_index('quantile').iloc[:,9:].transpose() for s in scenarios}

In [ ]:
scenarios

In [ ]:
years = np.arange(1850, 2351) 
D_out = evolve(np.zeros(9), A[0], len(years),
               tau_c = df_TEs['Collapse (years)'].to_numpy(),
               tau_r = df_TEs['Recovery (years)'].to_numpy(),
               T = dfs['ssp119']['mean'].to_numpy(),
               beta=df_TEs['Beta (1/°C)'].to_numpy(),
               T_lim = df_TEs['Threshold (°C) – Est.'].to_numpy())

In [ ]:
fig,axs = plt.subplots(figsize=(12,9),nrows=3,ncols=3,sharex=True,sharey=False)
fig.subplots_adjust(hspace=0.05) # hspace=0.05

prop_cycle = plt.rcParams['axes.prop_cycle']
colors = prop_cycle.by_key()['color']
# linestyles = [':',(0, (3, 1, 1, 1, 1, 1)),'-.','--', (2,(10,3)) ,(5,(10,3))]
# scen_labels = ['SSP1-1.9', 'SSP1-2.6', 'SSP2-4.5', 'SSP3-7.0', 'SSP5-3.4-over', 'SSP5-8.5']

# linestyles = [':',(0, (3, 1, 1, 1, 1, 1)),'-.','--',(5,(10,3))]
linestyles = [':'] + [(0,(2*j+2,1)) for j in range(len(scenarios)-2)] + ['-']
scen_labels = ['SSP1-1.9', 'SSP1-2.6', 'SSP2-4.5', 'SSP3-7.0', 'SSP5-8.5']
if len(scen_labels) != len(scenarios):
    scen_labels = scenarios
    
TEs = df_TEs.index.to_list()
TE_ordered = ['AMOC', 'WAM', 'REEF',
              'ENSO', 'AMAZ','PERM',
              'GIS', 'WAIS', 'ASI']
letters = ['(%s)'%l for l in 'abcdefghi']
TE_ind = {te:i for i,te in enumerate(TEs)}
for i_ax,te in enumerate(TE_ordered):
    i = TE_ind[te]
    axs[i_ax//3,i_ax%3].annotate(letters[i_ax] + ' '+ TEs[i],(0.05,0.95),
                                 xycoords='axes fraction',ha='left',va='top',fontsize=12,
                                 bbox=dict(fc='white',edgecolor='0.7',boxstyle='round'))
    # axs[i_ax//3,i_ax%3].plot(years[:D_hist.shape[-1]],D_hist[i,:,:].mean(axis=0)*100,
    #                    c=util.lighten_color(colors[i],0.65), 
    #                    ls='-',lw=1)
    axs[i_ax//3,i_ax%3].grid(visible=True,lw=0.5,axis='both')
    axs[i_ax//3,i_ax%3].plot(years,D_out[i,:]*100,
                                 c=util.lighten_color(colors[i],1),#0.1+(j+3)/(len(scenarios)+3)), 
                                 ls=linestyles[0],
                                 lw=1.5)

# lines=[Line2D([0], [0], color='k', lw=2, label='Historical')] 
# lines = [Line2D([0], [0], color=util.lighten_color('k',1),#0.1+(j+3)/(len(scenarios)+3)), 
#                 lw=2, 
#                 label=scen_labels[j], 
#                 ls = linestyles[j]) 
#          for j in range(len(scenarios))]

for i in range(3):
    axs[-1,i].set_xlabel('Year')
    axs[i,0].set_ylabel('Degraded prob. (%)')

In [ ]:
df_TEs.index.to_list()

# stationary distributions

In [14]:
stat_dist_theory(A[0],
                 1/(df_TEs['Collapse (years)'].to_numpy()+1),
                 1/(df_TEs['Collapse (years)'].to_numpy()+1), 
                 100, 
                 df_TEs['Threshold (°C) – Est.'].to_numpy(),
                 df_TEs['Beta (1/°C)']
                )

array([0.628153  , 0.84301122, 0.41659488, 0.65410108, 0.3021461 ,
       0.5493708 , 0.81824438, 0.72193739, 0.59122198])